# Week 10 · Topic 3 — Updating, Deleting and the Safe Pattern

**Box we are building: STORE, changed carefully**

The one mistake in this topic affects every row in a table at once — worth
slowing down for.

> **Already know this?** Skip to the *Topic 3 Challenge*.

> **Setup.** This notebook can be run on its own. The next cell connects to the same `network.db` file as Topic 1 and makes sure the `devices` table exists with two rows. It is safe to run more than once.

In [1]:
import sqlite3

conn = sqlite3.connect("network.db")
cursor = conn.cursor()
cursor.execute("""
    CREATE TABLE IF NOT EXISTS devices (
        id     INTEGER PRIMARY KEY,
        name   TEXT NOT NULL,
        ip     TEXT UNIQUE NOT NULL,
        active INTEGER DEFAULT 1,
        type   TEXT
    )
""")
cursor.execute("INSERT OR IGNORE INTO devices (name, ip) VALUES ('Router-A', '10.0.0.1')")
cursor.execute("INSERT OR IGNORE INTO devices (name, ip) VALUES ('Switch-B', '10.0.0.2')")
conn.commit()

---
## 1. `UPDATE` — changing existing rows

In [2]:
cursor.execute("UPDATE devices SET ip = ? WHERE id = ?", ("10.0.0.254", 1))
conn.commit()
print(cursor.execute("SELECT name, ip FROM devices WHERE id = 1").fetchone())

('Router-A', '10.0.0.254')


### Your turn
Update one device's name, then confirm the change with a SELECT.

In [4]:
# your code here
cursor.execute("UPDATE devices SET name = ? WHERE id = ?", ("Router-Main", 1))
conn.commit()
print(cursor.execute("SELECT id, name, ip FROM devices WHERE id = 1").fetchone())

(1, 'Router-Main', '10.0.0.254')


---
## 2. `rowcount` — how many rows actually changed

In [5]:
result = cursor.execute("UPDATE devices SET active = 0 WHERE name = ?", ("AP-Lobby",))
conn.commit()
print(result.rowcount)

0


### Your turn
Update a row and print how many rows `rowcount` says were affected.

In [6]:
# your code here
result = cursor.execute("UPDATE devices SET active = 0 WHERE id = ?", (1,))
conn.commit()
print(result.rowcount)

1


---
## 3. `DELETE`, and the mistake that affects every row

A missing `WHERE` on `UPDATE` or `DELETE` changes or removes **everything** in
the table — no confirmation, no undo.

In [7]:
cursor.execute("DELETE FROM devices WHERE active = 0")
conn.commit()
print(cursor.execute("SELECT name FROM devices").fetchall())

# cursor.execute("DELETE FROM devices")
# -> wipes the WHOLE table. Always run the matching SELECT first.

[('Switch-B',)]


### Your turn
Run the matching `SELECT` for a `DELETE` you are about to write, and check the result looks right, before actually deleting.

In [8]:
# your code here

print(cursor.execute("SELECT * FROM devices WHERE id = ?", (1,)).fetchall())
cursor.execute("DELETE FROM devices WHERE id = ?", (1,))
conn.commit()
print(cursor.execute("SELECT * FROM devices").fetchall())

[]
[(2, 'Switch-B', '10.0.0.2', 1, None)]


---
## 4. The `with` pattern — commit and close, handled for you

Auto-commits on success. Auto-rolls-back if an error is raised inside the
block — nothing half-finished gets saved.

In [ ]:
try:
    with sqlite3.connect("network.db") as c:
        c.execute("INSERT INTO devices (name, ip) VALUES (?, ?)", ("Duplicate", "10.0.0.254"))
except sqlite3.IntegrityError as e:
    print("Nothing was committed:", e)

### Your turn
Rewrite one of Topic 2's inserts to use the `with` pattern instead of a manual `commit()`.

In [9]:
# your code here
with sqlite3.connect("network.db") as c:
    c.execute("INSERT INTO devices (name, ip) VALUES (?, ?)", ("Switch-Floor2", "10.0.0.15"))

print(cursor.execute("SELECT * FROM devices WHERE name = ?", ("Switch-Floor2",)).fetchall())

[(3, 'Switch-Floor2', '10.0.0.15', 1, None)]


---
## Topic 3 Challenge

Using `with`, insert a device, then trigger a `UNIQUE` constraint violation
on purpose inside a second `with` block, and confirm the first insert is
still there afterwards but the second never landed.

In [10]:
# your code here
with sqlite3.connect("network.db") as c:
    c.execute("INSERT INTO devices (name, ip) VALUES (?, ?)", ("Firewall-Edge", "10.0.0.99"))

try:
    with sqlite3.connect("network.db") as c:
        c.execute("INSERT INTO devices (name, ip) VALUES (?, ?)", ("Duplicate-Device", "10.0.0.99"))
except sqlite3.IntegrityError as e:
    print("Rolled back second insert:", e)

print(cursor.execute("SELECT name, ip FROM devices WHERE ip = ?", ("10.0.0.99",)).fetchall())

Rolled back second insert: UNIQUE constraint failed: devices.ip
[('Firewall-Edge', '10.0.0.99')]


---
## Same idea, three fields

| Field | Looks like |
|---|---|
| **AI / Data Science** | correcting one bad value in a stored dataset |
| **Cyber Security** | marking one incident resolved, without touching the others |
| **IT** | decommissioning one server record, by id, never by a blanket delete |

---
## Common mistakes

| Mistake | What you see | Fix |
|---|---|---|
| `UPDATE`/`DELETE` with no `WHERE` | every row changes or disappears | always include `WHERE`, run the matching `SELECT` first |
| forgetting `conn.commit()` outside a `with` block | the change silently does not persist | commit explicitly, or switch to `with` |
| leaving a connection open | resources leak in a long-running program | close it, or use `with` so it happens automatically |
| assuming `rowcount` means something before `commit()` | it is still accurate — the mistake is elsewhere, usually a missing commit after | check commit happened, not just rowcount |

## Go deeper

`examples/01_update_and_delete.py` — UPDATE, DELETE, rowcount, the no-WHERE trap
`examples/02_the_with_pattern.py` — with, automatic rollback, sqlite3.Row

---
## Recap

| You can now | Using |
|---|---|
| change existing rows | `UPDATE ... SET ... WHERE ...` |
| remove rows | `DELETE FROM ... WHERE ...` |
| check what actually changed | `.rowcount` |
| commit and close safely | `with sqlite3.connect(...) as conn:` |

**Next:** Topic 4 — moving your real data and users into this.